![image.png](https://i.imgur.com/a3uAqnb.png)

# 📝 Semantic Search in a Refinery Manual

**Scenario:**
You have a 50-page refinery operations guide.
A technician types a question like *"pump vibration too high"*.
Can we find the most relevant pages using word embeddings?

**How it works:**
1. Turn each page into a **vector** (using GloVe)
2. Turn the question into a **vector**
3. Find pages closest to the question (cosine similarity)

**We will try TWO ways and compare:**
- ❌ Without preprocessing (raw text)
- ✅ With preprocessing (remove stop words + lemmatize)

> ⚠️ This exercise assumes GloVe is already loaded as `model` from the cells above.

In [ ]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 68.5 MB/s eta 0:00:00


In [ ]:
# Importing necessary libraries
import numpy as np  # Importing numpy for numerical operations
import matplotlib.pyplot as plt  # Importing matplotlib for plotting
from sklearn.decomposition import PCA  # Importing PCA for dimensionality reduction
from gensim.test.utils import datapath, get_tmpfile  # Importing utility functions from gensim
from gensim.models import KeyedVectors  # Importing KeyedVectors for handling word vectors
from gensim.scripts.glove2word2vec import glove2word2vec  # Importing function to convert GloVe format to word2vec
import gensim.downloader as api  # Importing gensim downloader to load pre-trained models

In [ ]:
# Load GloVe word vectors
# Gensim has functionality to download a pre-trained model and load it. We use the 100d vectors below.
model = api.load('glove-wiki-gigaword-100')  # Loading pre-trained GloVe vectors with 100 dimensions

[==================================================] 100.0% 128.1/128.1MB downloaded




---



## Step 1 — Load the Guide

In [ ]:
import pandas as pd
import numpy as np

guide = pd.read_excel("refinery_operations_guide.xlsx")

print(f"✅ Loaded {len(guide)} pages\n")
guide.head()

✅ Loaded 50 pages



,page_number,section,content
0,1,Equipment Overview,This chapter provides an overview of the major...
1,2,Centrifugal Pumps,Centrifugal pumps are the most common type of ...
2,3,Pump Maintenance,Routine pump maintenance includes vibration mo...
3,4,Pump Troubleshooting,"Common pump problems include cavitation, seal ..."
4,5,Compressors,Reciprocating and centrifugal compressors are ...


In [ ]:
# See all sections
for _, row in guide.iterrows():
    print(f"  Page {row['page_number']:>2} — {row['section']}")

  Page  1 — Equipment Overview
  Page  2 — Centrifugal Pumps
  Page  3 — Pump Maintenance
  Page  4 — Pump Troubleshooting
  Page  5 — Compressors
  Page  6 — Compressor Lubrication
  Page  7 — Heat Exchangers
  Page  8 — Heat Exchanger Cleaning
  Page  9 — Control Valves
  Page 10 — Electric Motors
  Page 11 — Crude Distillation Unit
  Page 12 — Vacuum Distillation Unit
  Page 13 — Hydrocracker Unit
  Page 14 — Diesel Hydrotreater
  Page 15 — Naphtha Hydrotreater
  Page 16 — Sulfur Recovery Unit
  Page 17 — Hydrogen Production Unit
  Page 18 — Amine Treating Unit
  Page 19 — Sour Water Stripper
  Page 20 — Tank Farm Operations
  Page 21 — Pressure Transmitters
  Page 22 — Level Transmitters
  Page 23 — Flow Transmitters
  Page 24 — Temperature Transmitters
  Page 25 — DCS and Control Systems
  Page 26 — Instrument Calibration
  Page 27 — Analyzer Systems
  Page 28 — Emergency Shutdown System
  Page 29 — Fire and Gas Detection
  Page 30 — H2S Safety
  Page 31 — Relief Valves
  Page 32 

In [ ]:
# Read one page
print(guide.loc[3, "content"])

Common pump problems include cavitation, seal leaks, high vibration, and bearing overheating. Cavitation occurs when the suction pressure drops below the vapor pressure of the fluid, causing bubbles to form and collapse inside the pump. Signs include a crackling noise and fluctuating discharge pressure.


## Step 2 — Tokenize the Text

In [ ]:
def simple_tokenize(text):
    return text.lower().split()

## Step 3 — Turn a Page into One Vector

In [ ]:
def page_vector(text, embeddings, tokenize_fn):
    # Split text into words
    words = tokenize_fn(text)

    # Look up each word in GloVe (skip unknown words)
    vectors = []
    for word in words:
        if word in embeddings:
            vectors.append(embeddings[word])

    # Average all vectors into one
    if len(vectors) == 0:
        return np.zeros(100)
    return np.mean(vectors, axis=0)

## Step 4 — Build Vectors for All 50 Pages

In [ ]:
raw_vectors = []

for text in guide["content"]:
    vec = page_vector(text, model, simple_tokenize)
    raw_vectors.append(vec)

raw_vectors = np.array(raw_vectors)

print(f"Shape: {raw_vectors.shape}")
print("(50 pages × 100 dimensions — each page is now one row of numbers!)")

Shape: (50, 100)
(50 pages × 100 dimensions — each page is now one row of numbers!)


## Step 5 — Search!
Given a question, we:
1. Turn the question into a vector (same way as pages)
2. Compare it to every page vector using **cosine similarity**
3. Return the top 5 most similar pages

Cosine similarity: **1.0** = identical, **0.0** = completely unrelated.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def search(query, page_vectors, tokenize_fn, top_n=5):
    # Turn query into a vector
    query_vec = page_vector(query, model, tokenize_fn)
    query_vec = query_vec.reshape(1, -1) # from (100,) to (1,100)

    # Compare with every page
    scores = cosine_similarity(query_vec, page_vectors)[0] # results come as [1,50] [[0.98,0.22..]], picking at [0] makes it [0.98,0.22..]

    # Sort by highest score
    top_indices = scores.argsort()[::-1][:top_n] #argsort make it min to max, [::-1] reverse it, :top_n returns top n results

    # Print results
    print(f'🔍 Query: "{query}"\n')
    for rank, idx in enumerate(top_indices, 1):
        page = guide.loc[idx, "page_number"]
        section = guide.loc[idx, "section"]
        score = scores[idx]
        print(f"   {rank}. Page {page:>2} — {section:<35s}  (score: {score:.3f})")

    return top_indices

In [ ]:
search("pump vibration is too high", raw_vectors, simple_tokenize)

🔍 Query: "pump vibration is too high"

   1. Page 30 — H2S Safety                           (score: 0.895)
   2. Page 47 — Steam System                         (score: 0.888)
   3. Page  3 — Pump Maintenance                     (score: 0.880)
   4. Page  4 — Pump Troubleshooting                 (score: 0.878)
   5. Page 17 — Hydrogen Production Unit             (score: 0.876)


array([29, 46,  2,  3, 16])

In [ ]:
search("H2S gas detected in the area", raw_vectors, simple_tokenize)

🔍 Query: "H2S gas detected in the area"

   1. Page 29 — Fire and Gas Detection               (score: 0.912)
   2. Page 19 — Sour Water Stripper                  (score: 0.895)
   3. Page 18 — Amine Treating Unit                  (score: 0.891)
   4. Page 16 — Sulfur Recovery Unit                 (score: 0.891)
   5. Page 32 — Flare System                         (score: 0.889)


array([28, 18, 17, 15, 31])

In [ ]:
search("how to calibrate a pressure transmitter", raw_vectors, simple_tokenize)

🔍 Query: "how to calibrate a pressure transmitter"

   1. Page 21 — Pressure Transmitters                (score: 0.919)
   2. Page 28 — Emergency Shutdown System            (score: 0.903)
   3. Page 44 — Abnormal Situation Management        (score: 0.902)
   4. Page 50 — Nitrogen and Instrument Air          (score: 0.901)
   5. Page  8 — Heat Exchanger Cleaning              (score: 0.897)


array([20, 27, 43, 49,  7])

## Step 6 — Why Preprocessing Might Help

Look at the query *"H2S gas detected in the area what to do"*:

```
["h2s", "gas", "detected", "in", "the", "area", "what", "to", "do"]
```

The words **"in", "the", "what", "to", "do"** are useless — they appear on every page.
They add noise and push the vector away from the right answer.

**Solution:** remove stop words and lemmatize before building vectors.

```
Before: ["h2s", "gas", "detected", "in", "the", "area", "what", "to", "do"]
After:  ["h2s", "gas", "detected", "area"]
```

Only the meaningful words remain!

In [ ]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


## ✏️ Task 1 — Complete the Clean Tokenizer

Fill in the **two blanks** (`___`) in the function below.

**Hints:**
- To check if a word is a stop word: `word in stop_words`
- To lemmatize a word: `lemmatizer.lemmatize(word)`

In [ ]:
def clean_tokenize(text):
    words = text.lower().split()
    cleaned = []
    for word in words:
        if ??? ???:       # skip numbers and punctuation
            continue
        if ????:                      # ← skip stop words
            continue
        word = ???                # ← lemmatize the word
        cleaned.append(word)
    return cleaned

**Test your function** — run the cell below. If it works, you should see the stop words removed and words like "pumps" → "pump".

In [ ]:
sample = "The pumps were leaking and operators isolated the equipment"

print("Raw:    ", simple_tokenize(sample))
print("Cleaned:", clean_tokenize(sample))

Raw:     ['the', 'pumps', 'were', 'leaking', 'and', 'operators', 'isolated', 'the', 'equipment']
Cleaned: ['pump', 'leaking', 'operator', 'isolated', 'equipment']


<details>
<summary>🔑 Click to see the solution</summary>

```python
def clean_tokenize(text):
    words = text.lower().split()
    cleaned = []
    for word in words:
        if not word.isalpha():
            continue
        if word in stop_words:              # skip stop words
            continue
        word = lemmatizer.lemmatize(word)    # lemmatize
        cleaned.append(word)
    return cleaned
```

</details>

## ✏️ Task 2 — Build Clean Vectors

Now build page vectors using your `clean_tokenize` function.

Fill in the **one blank** — which tokenizer should we use?


In [ ]:
clean_vectors = []

for text in guide[????]:
    vec = page_vector(text, model, ???)       # ← which tokenizer?
    clean_vectors.append(vec)

clean_vectors = np.array(clean_vectors)
print(f"Shape: {clean_vectors.shape}  ✅")

Shape: (50, 100)  ✅


<details>
<summary>🔑 Click to see the solution</summary>

```python
clean_vectors = []

for text in guide["content"]:
    vec = page_vector(text, model, clean_tokenize)
    clean_vectors.append(vec)

clean_vectors = np.array(clean_vectors)
print(f"Shape: {clean_vectors.shape}  ✅")
```

</details>

## ✏️ Task 3 — Search and Compare

Run the **same query** with both methods and compare.

Fill in the **one blank** to complete the second search.

**Hint:** use `clean_vectors` and `clean_tokenize`

In [ ]:
print("══ WITHOUT preprocessing ══")
search("pump vibration is too high", raw_vectors, simple_tokenize)

print()
print("══ WITH preprocessing ══")
search("pump vibration is too high", ??, ??)    # ← same query, but with clean_vectors and clean_tokenize

══ WITHOUT preprocessing ══
🔍 Query: "pump vibration is too high"

   1. Page 30 — H2S Safety                           (score: 0.895)
   2. Page 47 — Steam System                         (score: 0.888)
   3. Page  3 — Pump Maintenance                     (score: 0.880)
   4. Page  4 — Pump Troubleshooting                 (score: 0.878)
   5. Page 17 — Hydrogen Production Unit             (score: 0.876)

══ WITH preprocessing ══


''

Now try the H2S query:

In [ ]:
print("══ WITHOUT preprocessing ══")
search("H2S gas detected in the area what to do", raw_vectors, simple_tokenize)

print()
print("══ WITH preprocessing ══")
search("H2S gas detected in the area what to do", ??, ??)

══ WITHOUT preprocessing ══
🔍 Query: "H2S gas detected in the area what to do"

   1. Page 29 — Fire and Gas Detection               (score: 0.939)
   2. Page 30 — H2S Safety                           (score: 0.938)
   3. Page 32 — Flare System                         (score: 0.937)
   4. Page 28 — Emergency Shutdown System            (score: 0.927)
   5. Page 44 — Abnormal Situation Management        (score: 0.927)

══ WITH preprocessing ══
🔍 Query: "H2S gas detected in the area what to do"

   1. Page 29 — Fire and Gas Detection               (score: 0.890)
   2. Page  6 — Compressor Lubrication               (score: 0.796)
   3. Page 32 — Flare System                         (score: 0.796)
   4. Page 22 — Level Transmitters                   (score: 0.781)
   5. Page 16 — Sulfur Recovery Unit                 (score: 0.776)


array([28,  5, 31, 21, 15])

## Step 7 — Measure with Ground Truth

So far we've been eyeballing results. Let's measure properly.

We have **10 queries with correct answers** — the pages a refinery engineer would actually want.
We count: **how many correct pages does each method find in its top 5?**

In [ ]:
queries_with_answers = [
    {
        "query": "pump seal is leaking oil",
        "golden_pages": [2, 3, 4],
        "why": "Centrifugal Pumps, Pump Maintenance, Pump Troubleshooting",
    },
    {
        "query": "H2S gas detected in the area what to do",
        "golden_pages": [30, 29, 33],
        "why": "H2S Safety, Fire and Gas Detection, PPE",
    },
    {
        "query": "how to calibrate a pressure transmitter",
        "golden_pages": [21, 26],
        "why": "Pressure Transmitters, Instrument Calibration",
    },
    {
        "query": "relief valve opened and gas is going to flare",
        "golden_pages": [31, 32],
        "why": "Relief Valves, Flare System",
    },
    {
        "query": "electric motor keeps tripping the breaker",
        "golden_pages": [10, 35],
        "why": "Electric Motors, LOTO Procedures",
    },
    {
        "query": "heat exchanger pressure drop is increasing",
        "golden_pages": [7, 8],
        "why": "Heat Exchangers, Heat Exchanger Cleaning",
    },
    {
        "query": "what PPE is required before entering the plant",
        "golden_pages": [33, 34],
        "why": "PPE, Permit to Work",
    },
    {
        "query": "steps to shut down the unit in an emergency",
        "golden_pages": [28, 42],
        "why": "Emergency Shutdown System, Unit Shutdown Procedure",
    },
    {
        "query": "corrosion found on a pipeline during inspection",
        "golden_pages": [36, 37, 38],
        "why": "Corrosion Mechanisms, Inspection Techniques, Pipeline Integrity",
    },
    {
        "query": "compressor oil sample shows metal particles",
        "golden_pages": [5, 6],
        "why": "Compressors, Compressor Lubrication",
    },
]


In [ ]:
def count_hits(query, golden_pages, page_vectors, tokenize_fn):
    query_vec = page_vector(query, model, tokenize_fn).reshape(1, -1)
    scores = cosine_similarity(query_vec, page_vectors)[0]
    top_indices = scores.argsort()[::-1][:5]
    top_pages = [guide.loc[idx, "page_number"] for idx in top_indices]
    hits = sum(1 for p in golden_pages if p in top_pages)
    return hits, len(golden_pages), top_pages


# ── Score all 10 queries ──────────────────────────
print(f"{'#':<4} {'Query':<50} {'Raw':>6} {'Clean':>6}")
print("-" * 70)

raw_total = 0
clean_total = 0
possible_total = 0

for i, q in enumerate(queries_with_answers, 1):
    raw_hits, possible, _   = count_hits(q["query"], q["golden_pages"],
                                          raw_vectors, simple_tokenize)
    clean_hits, _, _        = count_hits(q["query"], q["golden_pages"],
                                          clean_vectors, clean_tokenize)

    raw_total += raw_hits
    clean_total += clean_hits
    possible_total += possible

    short = q["query"][:48]
    print(f"{i:<4} {short:<50} {raw_hits}/{possible:>2}   {clean_hits}/{possible:>2}")

print("-" * 70)
print(f"{'':4} {'TOTAL':<50} {raw_total}/{possible_total:>2}   {clean_total}/{possible_total:>2}")
print()
print(f"🏆 WITHOUT preprocessing:  {raw_total}/{possible_total} correct pages found in top 5")
print(f"🏆 WITH    preprocessing:  {clean_total}/{possible_total} correct pages found in top 5")

#    Query                                                 Raw  Clean
----------------------------------------------------------------------
1    pump seal is leaking oil                           0/ 3   0/ 3
2    H2S gas detected in the area what to do            2/ 3   1/ 3
3    how to calibrate a pressure transmitter            1/ 2   1/ 2
4    relief valve opened and gas is going to flare      2/ 2   2/ 2
5    electric motor keeps tripping the breaker          1/ 2   1/ 2
6    heat exchanger pressure drop is increasing         1/ 2   1/ 2
7    what PPE is required before entering the plant     1/ 2   2/ 2
8    steps to shut down the unit in an emergency        2/ 2   2/ 2
9    corrosion found on a pipeline during inspection    2/ 3   2/ 3
10   compressor oil sample shows metal particles        1/ 2   1/ 2
----------------------------------------------------------------------
     TOTAL                                              13/23   13/23

🏆 WITHOUT preprocessing:  13/23 corre

## Step 8 — Look at One Query in Detail

Change `QUERY_NUMBER` below to explore different queries (1 to 10).

Look at:
- Which pages got ✅ (correct) and which didn't
- The "words used for matching" at the bottom — what did preprocessing remove?

In [ ]:
QUERY_NUMBER = 2       # ← change this (1 to 10)

q = queries_with_answers[QUERY_NUMBER - 1]
print(f'Query: "{q["query"]}"')
print(f'Correct pages: {q["golden_pages"]} ({q["why"]})\n')

print("── WITHOUT preprocessing ──")
_, _, raw_top = count_hits(q["query"], q["golden_pages"], raw_vectors, simple_tokenize)
for rank, p in enumerate(raw_top, 1):
    hit = "✅" if p in q["golden_pages"] else "  "
    print(f"   {rank}. Page {p:>2} — {guide.loc[p-1, 'section']}  {hit}")

print("\n── WITH preprocessing ──")
_, _, clean_top = count_hits(q["query"], q["golden_pages"], clean_vectors, clean_tokenize)
for rank, p in enumerate(clean_top, 1):
    hit = "✅" if p in q["golden_pages"] else "  "
    print(f"   {rank}. Page {p:>2} — {guide.loc[p-1, 'section']}  {hit}")

print("\n── Words used for matching ──")
print("   Raw:    ", simple_tokenize(q["query"]))
print("   Cleaned:", clean_tokenize(q["query"]))

Query: "H2S gas detected in the area what to do"
Correct pages: [30, 29, 33] (H2S Safety, Fire and Gas Detection, PPE)

── WITHOUT preprocessing ──
   1. Page 29 — Fire and Gas Detection  ✅
   2. Page 30 — H2S Safety  ✅
   3. Page 32 — Flare System    
   4. Page 28 — Emergency Shutdown System    
   5. Page 44 — Abnormal Situation Management    

── WITH preprocessing ──
   1. Page 29 — Fire and Gas Detection  ✅
   2. Page  6 — Compressor Lubrication    
   3. Page 32 — Flare System    
   4. Page 22 — Level Transmitters    
   5. Page 16 — Sulfur Recovery Unit    

── Words used for matching ──
   Raw:     ['h2s', 'gas', 'detected', 'in', 'the', 'area', 'what', 'to', 'do']
   Cleaned: ['gas', 'detected', 'area']


## ❓ Questions

1. Which method got a **higher total score**? By how many hits?

2. Set `QUERY_NUMBER = 2` (H2S query). How many words did preprocessing remove? Were those words useful or just noise?

3. Set `QUERY_NUMBER = 8` (emergency shutdown). Did the ranking of correct pages change?

4. We are **averaging** all word vectors into one vector. What information do we lose?
   - Does *"pump failed"* have the same vector as *"failed pump"*?
   - Does a 10-word page get treated the same as a 200-word page?

5. If a technician searches *"the unit is not running"*, the word *"not"* is a stop word and gets removed. Could that be a problem?

### Contributed by : Yazan alshoibi